In [117]:
from urllib.request import urlopen
import pandas as pd

import re

import dask.bag as db
import json

In [95]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [99]:
%cd /content/drive/MyDrive/ФУППРФ/5 семестр/Технологии обработки больших данных/data/12

/content/drive/MyDrive/ФУППРФ/5 семестр/Технологии обработки больших данных/data/12


# Dask Bag

Материалы:
* Макрушин С.В. Лекция 12: Map-Reduce
* https://docs.dask.org/en/latest/bag.html
* JESSE C. DANIEL. Data Science with Python and Dask.

## Задачи для совместного разбора

1. Считайте файл `Dostoevskiy Fedor. Igrok - BooksCafe.Net.txt` и разбейте на предложения. Подсчитайте длину (в кол-ве символов) каждого предложения.

In [70]:
def download_file_url(file_id: str):
    return f'https://drive.google.com/uc?id={file_id}'


book_url = download_file_url('1YKvq9yRcP14LbujLn9C9V_z60TfoZDDu')
with urlopen(book_url) as response:
    text = response.read().decode('cp1251').replace('\r\n', '').replace('\xa0', '').replace(' (франц.)', '')
    sentences = re.split(r'([А-ЯЁ].+?[\.\?!]+)', text)[1:-1]

sent_df = pd.DataFrame(list(map(lambda s: (len(s), s), sentences)), columns=['n_symbols', 'sentence'])
sent_df = sent_df[~((sent_df.n_symbols >= 0) & (sent_df.n_symbols < 4))]
sent_df

,n_symbols,sentence
0,73,"Спасибо, что скачали книгу в бесплатной электр..."
1,25,Net: http://bookscafe.net
2,35,Все книги автора: http://bookscafe.
3,38,net/author/dostoevskiy_fedor-1096.html
4,49,Эта же книга в других форматах: http://bookscafe.
...,...,...
6927,93,95но видишь ли… госпожа генеральша эти дьяволь...
6928,39,Заго-Заго и еще четырнадцать согласных!
6929,34,"как это приятно, не правда ли?.96"
6930,45,"Я считала тебя глупым, и ты смотрел дурачком."


2. Считайте файл `Dostoevskiy Fedor. Igrok - BooksCafe.Net.txt` и разбейте на предложения. Выведите предложения, длина которых не более 10 символов.

In [71]:
sent_df[sent_df['n_symbols'] <= 10]

,n_symbols,sentence
144,8,Плюнуть?
156,7,России.
176,4,Как!
290,7,Полина.
318,7,Почему?
...,...,...
6858,7,Четыре.
6863,10,(нем.).66
6864,8,Это он!!
6914,10,"О, но ты…."


3. На основе списка предложений из задачи 1-2 создайте `dask.bag`. Рассчитайте среднюю длину предложений в тексте.

In [76]:
sent_bag = db.from_sequence(sentences)
sent_bag.map(len).mean().compute()

40.884609837011396

4. На основе файла `addres_book.json` создайте `dask.bag`. Посчитайте количество мобильных и рабочих телефонов в наборе данных

In [128]:
with open('addres-book.json') as f:
    d = json.load(f)

addres_bag = db.from_sequence(d)
addres_bag

dask.bag<from_sequence, npartitions=8>

In [136]:
addres_bag.pluck('phones').map(len).sum().compute()

13

## Лабораторная работа 12

1. В файлах архиве `reviews_full.zip` находятся файлы, содержащие информацию об отзывах к рецептам в формате JSON Lines. Отзывы разделены на файлы в зависимости от оценки (например, в файле `reviews_1.json` находятся отзывы с оценкой 1). Считайте файлы из этого архива в виде `dask.bag`. Преобразуйте текстовое содержимое файлов в объекты python (с помощью модуля `json`). Выведите на экран первые 5 элементов полученного `bag`.

In [174]:
records = db.read_text([f'reviews_{i}.json' for i in range(0, 6)])
records

dask.bag<bag-from-delayed, npartitions=6>

In [175]:
records.map(json.loads).take(5)

({'user_id': 452355,
  'recipe_id': 292657,
  'date': '2016-05-08',
  'review': 'WOW!!! This is the best. I have never been able to make homemade enchiladas that taste like the Mexican restaurants. I made this last night for my family and they said they will never have enchiladas at the Mexican Restaurants again. Thanks for sharing.'},
 {'user_id': 329304,
  'recipe_id': 433404,
  'date': '2006-06-14',
  'review': 'This was good but the dressing needed something and I found it to be a little too sweet, next time I will experiment with some garlic and herbs and reduce the sugar slightly, thanks for sharing kcdlong!...Kitten'},
 {'user_id': 227932,
  'recipe_id': 2008187,
  'date': '1985-11-19',
  'review': 'Very good,it was a hit for my family. I used 6 cloves of garlic and had 1 lb beef and  Johnsonville sausage,1/2 lb hot and  1/2 lb honey garlic( which I wanted to use). That was a perfect combo for us. The sausage gave it nice flavor No guestion , I will be making this often.'},
 {'u

2. Модифицируйте функцию разбора JSON таким образом, чтобы в каждый словарь c информацией об отзыве добавить ключ `rating`. Значение получите на основе названия файла (см. аргумент `include_path`), использовав для этого регулярное выражение.

In [232]:
def parse_json(tup):
    json_, file_name = tup
    d = json.loads(json_)
    d['rating'] = int(re.findall(r'reviews_(\d)+', file_name)[0])
    return d
records = db.read_text('reviews_*.json', include_path=True)
records

dask.bag<bag-from-delayed, npartitions=6>

In [233]:
records_parsed = records.map(parse_json)
records_parsed.take(4)

({'user_id': 452355,
  'recipe_id': 292657,
  'date': '2016-05-08',
  'review': 'WOW!!! This is the best. I have never been able to make homemade enchiladas that taste like the Mexican restaurants. I made this last night for my family and they said they will never have enchiladas at the Mexican Restaurants again. Thanks for sharing.',
  'rating': 0},
 {'user_id': 329304,
  'recipe_id': 433404,
  'date': '2006-06-14',
  'review': 'This was good but the dressing needed something and I found it to be a little too sweet, next time I will experiment with some garlic and herbs and reduce the sugar slightly, thanks for sharing kcdlong!...Kitten',
  'rating': 0},
 {'user_id': 227932,
  'recipe_id': 2008187,
  'date': '1985-11-19',
  'review': 'Very good,it was a hit for my family. I used 6 cloves of garlic and had 1 lb beef and  Johnsonville sausage,1/2 lb hot and  1/2 lb honey garlic( which I wanted to use). That was a perfect combo for us. The sausage gave it nice flavor No guestion , I will

3. Посчитайте количество отзывов в исходном датасете.

In [180]:
records.count().compute()

9057540

4. Отфильтруйте `bag`, сохранив только отзывы, оставленные в 2014 и 2015 годах.

In [234]:
records_parsed_14_15 = records_parsed.filter(lambda d: d['date'].startswith('2014') or d['date'].startswith('2015'))

In [235]:
records_parsed_14_15.take(10)

({'user_id': 229850,
  'recipe_id': 1300038,
  'date': '2014-10-03',
  'review': 'Took this to a New Year&#039;s Eve Party. Everyone loved it! It&#039;s absolutely perfect, the flavor, the crunch, just delicious!',
  'rating': 0},
 {'user_id': 2706705,
  'recipe_id': 133747,
  'date': '2015-05-08',
  'review': 'Simple and easy way to enjoy a slice of pizza any time!  Well-toasted bread is the key - really toast it!  I put a bit of pizza sauce underneath my cheese for a more pizza-like flavor.  I used sourdough bread & medium cheddar cheese.  Fast & fun!  Great idea!  Made for 1-2-3 Hits Tag Game.',
  'rating': 0},
 {'user_id': 945545,
  'recipe_id': 898468,
  'date': '2015-06-30',
  'review': 'Delish!  I wanted to make this spicy so I used hot enchilada sauce and jalapeno refried beans.  I forgot to buy the onions so I doctored up the beans with onion powder and granulated garlic.  Added the olives under the cheese and baked, uncovered, for the 25 minutes.  Served with pico de gallo, s

In [236]:
records_parsed_14_15.count().compute()

735274

In [237]:
records_parsed_14_15.take(3)

({'user_id': 229850,
  'recipe_id': 1300038,
  'date': '2014-10-03',
  'review': 'Took this to a New Year&#039;s Eve Party. Everyone loved it! It&#039;s absolutely perfect, the flavor, the crunch, just delicious!',
  'rating': 0},
 {'user_id': 2706705,
  'recipe_id': 133747,
  'date': '2015-05-08',
  'review': 'Simple and easy way to enjoy a slice of pizza any time!  Well-toasted bread is the key - really toast it!  I put a bit of pizza sauce underneath my cheese for a more pizza-like flavor.  I used sourdough bread & medium cheddar cheese.  Fast & fun!  Great idea!  Made for 1-2-3 Hits Tag Game.',
  'rating': 0},
 {'user_id': 945545,
  'recipe_id': 898468,
  'date': '2015-06-30',
  'review': 'Delish!  I wanted to make this spicy so I used hot enchilada sauce and jalapeno refried beans.  I forgot to buy the onions so I doctored up the beans with onion powder and granulated garlic.  Added the olives under the cheese and baked, uncovered, for the 25 minutes.  Served with pico de gallo, s

5. Выполните препроцессинг отзывов:
    * привести строки к нижнему регистру
    * обрезать пробельные символы в начале и конце строки
    * удалите все символы, кроме английских букв и пробелов
    
Примените препроцессинг ко всем записям из `bag`, полученного в задании 4.

In [238]:
def preprocess(d):
    if 'review' in d:
        d['review'] = d['review'].lower().strip()
        d['review'] = re.sub(r'[^a-z\s]', '', d['review'])
    return d

In [239]:
records_prep = records_parsed_14_15.map(preprocess)

In [240]:
records_prep.take(4)

({'user_id': 229850,
  'recipe_id': 1300038,
  'date': '2014-10-03',
  'review': 'took this to a new years eve party everyone loved it its absolutely perfect the flavor the crunch just delicious',
  'rating': 0},
 {'user_id': 2706705,
  'recipe_id': 133747,
  'date': '2015-05-08',
  'review': 'simple and easy way to enjoy a slice of pizza any time  welltoasted bread is the key  really toast it  i put a bit of pizza sauce underneath my cheese for a more pizzalike flavor  i used sourdough bread  medium cheddar cheese  fast  fun  great idea  made for  hits tag game',
  'rating': 0},
 {'user_id': 945545,
  'recipe_id': 898468,
  'date': '2015-06-30',
  'review': 'delish  i wanted to make this spicy so i used hot enchilada sauce and jalapeno refried beans  i forgot to buy the onions so i doctored up the beans with onion powder and granulated garlic  added the olives under the cheese and baked uncovered for the  minutes  served with pico de gallo sour cream and avocado chunks  fantastic  tha

6. Посчитайте количество отзывов в датасете, полученном в результате решения задачи 5. В случае ошибок прокомментируйте результат и исправьте функцию препроцессинга.

In [241]:
records_prep.count().compute()

735274

7. Посчитайте, как часто в наборе, полученном в задании 5, встречается та или иная оценка

In [242]:
records_prep.pluck('rating').frequencies().compute()

[(0, 42472), (1, 9246), (2, 9380), (3, 26532), (4, 119413), (5, 528231)]

8. Найдите среднее значение `rating` в выборке

In [243]:
records_prep.pluck('rating').mean().compute()

4.388036296673077

9. Используя метод `foldby`, подсчитать максимальную длину отзывов в зависимости от оценки `rating` в наборе, полученном в задании 5.

In [247]:
records_prep.foldby('rating',
                    lambda prev, curr: max(prev, len(curr['review'])),
                    initial=0,
                    combine=max,
                    combine_initial=0).compute()

[(0, 6548), (1, 2868), (2, 2844), (3, 3174), (4, 6548), (5, 5351)]